# Gait-ViViT: A Video Processing Model for Parkinson's Disease Detection

In [ ]:
# Required libraries.
from transformers import VivitModel, VivitForVideoClassification
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3 - Model Implementation

The model will be implemented from scratch based on the structure of [Arnab et al.'s *Video Vision Transformer*](https://arxiv.org/abs/2103.15691), which can be studied using the `VivitForVideoClassification` class provided by [Hugging Face](https://huggingface.co/docs/transformers/en/model_doc/vivit).

In [ ]:
# Analyse the structure of the Video Vision Transformer model.
model = VivitModel.from_pretrained("google/vivit-b-16x2-kinetics400", attn_implementation="sdpa", device_map="auto") # By default, this instance uses scaled dot-product attention and maps the model on available GPUs.
print(model)

In [ ]:
# Analyse the structure of the Video Vision Transformer model.
model = VivitForVideoClassification.from_pretrained("google/vivit-b-16x2-kinetics400", attn_implementation="sdpa", device_map="auto") # By default, this instance uses scaled dot-product attention and maps the model on available GPUs.
print(model)

### Tubelet Embeddings

The tubelet embedding method can be seen as an extension of the *Vision Transformer*'s embedding method for video data.

In fact, the idea is to **extract non-overlapping patches** of size $t \times h \times w$, where $t$ is known as the "tubelet size" while $(h, w)$ is the spatial resolution of each patch, and **linearly project** them to an embedding dimension $\mathbb{R}^d$ using a **3D convolution**.

Therefore, for a tensor of shape $(C, T, H, W)$, the number of extracted tokens will be equal to $N = \lfloor\frac{T}{t}\rfloor \cdot \lfloor\frac{H}{h}\rfloor \cdot \lfloor\frac{W}{w}\rfloor$.

**N.B.:** From now on, let $n_t = \lfloor\frac{T}{t}\rfloor$, $n_h = \lfloor\frac{H}{h}\rfloor$ and $n_w = \lfloor\frac{W}{w}\rfloor$.

In [ ]:
class VivitTubeletEmbeddings(nn.Module):
  def __init__(self, img_size=(224, 224), patch_size=(16, 16), num_frames=32, tubelet_size=2, in_channels=3, embed_dim=768):
    super().__init__()

    # Define the parameters, using the default values from the original paper.
    self.img_size = img_size
    self.patch_size = patch_size
    self.num_frames = num_frames
    self.tubelet_size = tubelet_size

    # Determine the spatial and temporal dimensions of each patch and determine the number of extracted tokens.
    self.num_spatial_patches = (img_size[0] // patch_size[0]) * (img_size[1] // patch_size[1]) # (H // h) * (W // w).
    self.num_temporal_patches = num_frames // tubelet_size # T // t.
    self.total_patches = self.num_spatial_patches * self.num_temporal_patches # N = (H // h) * (W // w) * (T // t)

    # Extract and project non-overlapping patches using a 3D convolution.
    self.proj = nn.Conv3d(
        in_channels=in_channels,
        out_channels=embed_dim,
        kernel_size=(tubelet_size, patch_size[0], patch_size[1]),
        stride=(tubelet_size, patch_size[0], patch_size[1])
    )

  def forward(self, x):
    # Take a tensor of shape (B, C, T, H, W) and project it to (B, embed_dim, T // t, H // h, W // w) using a 3D convolution.
    x = self.proj(x)

    # Flatten the tensor from (B, embed_dim, T // t, H // h, W // w) to (B, embed_dim, N), where N = (T // t) * (H // h) * (W // w).
    x = x.flatten(2) # This tells the block to flatten from dimension 2 onwards.

    # Reshape the tensor from (B, embed_dim, N) to (B, N, embed_dim) to make it compatible with the model.
    x = x.transpose(1, 2)

    return x

In [ ]:
class VivitEmbeddings(nn.Module):
  def __init__(self, img_size=(224, 224), patch_size=(16, 16), num_frames=32, tubelet_size=2, in_channels=3, embed_dim=768, dropout=0.0):
    super().__init__()
    self.emb = VivitTubeletEmbeddings(img_size=img_size, patch_size=patch_size, num_frames=num_frames, tubelet_size=tubelet_size, in_channels=in_channels, embed_dim=embed_dim)
    self.drop = nn.Dropout(dropout)

  def forward(self, x):
    # Apply the tubelet embedding mechanism.
    x = self.emb(x)

    # Apply the dropout mask.
    x = self.drop(x)

    return x

### Transformer Encoder Block

A single encoder block alternates between **computing self-attention** and a **multi-layer perceptron**.
1. On an input $\mathbf{X} \in \mathbb{R}^{N \times d}$, the self-attention mechanism extracts the queries $\mathbf{Q} = \mathbf{XW}_q$, keys $\mathbf{K} = \mathbf{XW}_k$ and values $\mathbf{V} = \mathbf{XW}_v$ by a linear projection on the input and uses these vectors to compute the attention scores
$$
\mathbf{Y} = \text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Softmax}\bigg(\frac{\mathbf{QK}^T}{\sqrt{d_k}}\bigg)\mathbf{V}
$$
2. After computing the attention scores, the sequence is passed into a **multilayer perceptron** whose structure follows the one implemented in the *Vision Transformer* backbone used for the original model.

In [ ]:
class VivitAttention(nn.Module):
  def __init__(self, num_spatial_patches, num_temporal_patches, embed_dim=768, num_heads=12, qkv_bias=True, dropout=0.0):
    super().__init__()

    # Define the parameters.
    self.num_spatial_patches = num_spatial_patches
    self.num_temporal_patches = num_temporal_patches
    self.num_heads = num_heads
    self.head_dim = embed_dim // num_heads # Defined to simplify computations in the forward method.

    # Extract queries, keys and values using a linear projection with additive bias.
    self.q_proj = nn.Linear(embed_dim, embed_dim, bias=qkv_bias)
    self.k_proj = nn.Linear(embed_dim, embed_dim, bias=qkv_bias)
    self.v_proj = nn.Linear(embed_dim, embed_dim, bias=qkv_bias)
    self.attn_drop = nn.Dropout(dropout)
    self.proj = nn.Linear(embed_dim, embed_dim, bias=qkv_bias)
    self.proj_drop = nn.Dropout(dropout)

  def forward(self, x, num_spatial_patches, num_temporal_patches):
    B, N, D = x.shape # Note: N = 1 + (num_spatial_patches * num_temporal_patches), D = embed_dim.

    q = self.q_proj(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # Compute the queries.
    k = self.k_proj(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # Compute the keys.
    v = self.v_proj(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # Compute the values.

    x = F.scaled_dot_product_attention(q, k, v)
    x = x.transpose(1, 2).reshape(B, N, D)
    x = self.proj(x)
    x = self.proj_drop(x)

    return x

In [ ]:
class VivitMLP(nn.Module):
  def __init__(self, in_features, hidden_features, dropout=0.0):
    super().__init__()
    self.fc1 = nn.Linear(in_features, hidden_features) # Project from in_features to hidden_features.
    #self.act = nn.GELU(approximate="tanh") # Set approximate="tanh" for the FastGELU approximation.
    self.fc2 = nn.Linear(hidden_features, in_features) # Project from hidden_features to in_features.
    self.drop = nn.Dropout(dropout)

  def forward(self, x):
    x = self.fc1(x)
    x = 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3.0))))
    x = self.drop(x)
    x = self.fc2(x)
    x = self.drop(x)
    return x

In [ ]:
class VivitLayer(nn.Module):
  def __init__(self, num_spatial_patches, num_temporal_patches, embed_dim, num_heads, mlp_ratio=4.0, qkv_bias=True, dropout=0.0):
    super().__init__()

    # Define the parameters.
    self.num_spatial_patches = num_spatial_patches
    self.num_temporal_patches = num_temporal_patches

    # Apply layer normalization and compute attention scores.
    self.norm1 = nn.LayerNorm(embed_dim, eps=1e-6, elementwise_affine=True) # Use the settings indicated in the original model.
    self.attn = VivitAttention(self.num_spatial_patches, self.num_temporal_patches, embed_dim, num_heads=num_heads, qkv_bias=qkv_bias, dropout=dropout)
    self.norm2 = nn.LayerNorm(embed_dim, eps=1e-6, elementwise_affine=True) # Use the settings indicated in the original model.

    # Pass through the multilayer perceptron.
    self.hidden_features = int(embed_dim * mlp_ratio) # Chosen according to the original backbone.
    self.mlp = VivitMLP(in_features=embed_dim, hidden_features=self.hidden_features, dropout=dropout)
    self.drop = nn.Dropout(dropout)

  def forward(self, x, num_spatial_patches, num_temporal_patches):
    x = x + self.attn(self.norm1(x), num_spatial_patches, num_temporal_patches) # x = x + MSA(LN(x)).
    x = x + self.mlp(self.norm2(x)) # x = x + MLP(LN(x)).
    return x

### Adapting the Classification Head

While the original *Video Vision Transformer* backbone was implemented for multiclass classification, this new model implements **binary classification for anomaly detection**, requiring to adjust the classification head for this new task.

For this reason, the `AnomalyHead` class simply features a linear layer that upscales the features before processing them with an activation function, ultimately projecting the result onto a one-dimensional logit.

In [ ]:
class AnomalyHead(nn.Module):
  def __init__(self, in_features, hidden_features):
    super().__init__()

    self.fc1 = nn.Linear(in_features, hidden_features)
    self.act = nn.ReLU() # Used in Versions 1-3 of the model.
    # self.act = nn.GELU() # Used in Versions 4-6 of the model.
    self.fc2 = nn.Linear(hidden_features, 1)

  def forward(self, x):
    x = self.fc1(x)
    x = self.act(x)
    logit = self.fc2(x)
    return logit

### The Complete Model

The overall structure of the model will be closely based on the original *Video Vision Transformer* architecture used for the `VivitForVideoClassificationClass`.

In fact, the model first extracts non-overlapping patch embeddings from the input tensor and later prepends the `[CLS]` token to the embedding sequence, also adding the positional embeddings.

Next, the model passes the sequence to the transformer encoder, which consists of 12 repeated blocks in accordance to the original architecture.

Lastly, the model applies layer normalization and extracts the representation of the `[CLS]` token, which will be used by the final classification head to compute the logit for the input tensor.

In [ ]:
class GaitViViT(nn.Module):
  def __init__(self, img_size=(224, 224), patch_size=(16, 16), num_frames=32, tubelet_size=2, in_channels=3, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, qkv_bias=True, cls_ratio=2.0, dropout=0.0):
    super().__init__()

    # Extract the embeddings and determine the number of spatial and temporal batches.
    self.patch_embed = VivitEmbeddings(
        img_size=img_size,
        patch_size=patch_size,
        num_frames=num_frames,
        tubelet_size=tubelet_size,
        in_channels=in_channels,
        embed_dim=embed_dim,
        dropout=dropout
        )
    self.num_spatial_patches = self.patch_embed.emb.num_spatial_patches
    self.num_temporal_patches = self.patch_embed.emb.num_temporal_patches

    # Create and initialize the [CLS] token and the patch embeddings.
    self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
    self.pos_embed = nn.Parameter(torch.zeros(1, 1 + self.num_spatial_patches * self.num_temporal_patches, embed_dim))
    nn.init.normal_(self.cls_token, std=0.02) # Initialize from a Gaussian distribution with mean = 0 and standard deviation = 0.02.
    nn.init.normal_(self.pos_embed, std=0.02) # Initialize from a Gaussian distribution with mean = 0 and standard deviation = 0.02.

    self.pos_drop = nn.Dropout(dropout)

    # Create the transformer encoder block using the VivitLayer class.
    self.blocks = nn.ModuleList([
        VivitLayer(
            num_spatial_patches=self.num_spatial_patches,
            num_temporal_patches=self.num_temporal_patches,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias,
            dropout=dropout
        )
        for _ in range(depth)
    ])

    # Apply layer normalization before entering the classification head.
    self.norm = nn.LayerNorm(embed_dim, eps=1e-6, elementwise_affine=True) # Use the settings indicated in the original model.

    # Enter the classification head.
    self.head_hf = int(embed_dim * cls_ratio)
    self.head = AnomalyHead(in_features=embed_dim, hidden_features=self.head_hf)

  def forward(self, x):
    # Determine the batch size.
    B = x.shape[0] # Remember that x will have shape (B, C, T, H, W)

    # Extract patch embeddings.
    x = self.patch_embed(x)

    # Prepend the classification token to the patch sequence and add the positional embeddings.
    cls_tokens = self.cls_token.expand((B, -1, -1)) # Expand from (1, 1, embed_dim) to (B, 1, embed_dim), with -1 indicating not to change the size of that dimension.
    x = torch.cat([cls_tokens, x], dim=1) # Concatenate to shape (B, 1 + num_spatial_patches * num_temporal_patches, embed_dim).
    x = x + self.pos_embed
    x = self.pos_drop(x)

    # Pass through the transformer encoder.
    for block in self.blocks:
      x = block(x, self.num_spatial_patches, self.num_temporal_patches) # Remember that the forward function of the block requires specifying the number of patches as well.

    x = self.norm(x)

    # Extract the classification token that will be passed to the classification head.
    cls_output = x[:, 0, :] # Collapse to shape (B, embed_dim).
    logit = self.head(cls_output)

    return logit

### Model Initialization

After creating an instance of the model, it is possible to **load the pre-trained weights for the backbone**.

However, since weight names might differ, this procedure requires **mapping these weights onto the corresponding weights in the custom model**.

In [ ]:
def weight_map(hf_state_dict):
  new_state_dict = {}

  # Map weights for the [CLS] token and the positional embeddings.
  new_state_dict["cls_token"] = hf_state_dict["vivit.embeddings.cls_token"]
  new_state_dict["pos_embed"] = hf_state_dict["vivit.embeddings.position_embeddings"]

  # Map weights for the patch embeddings.
  new_state_dict["patch_embed.emb.proj.weight"] = hf_state_dict["vivit.embeddings.patch_embeddings.projection.weight"]
  new_state_dict["patch_embed.emb.proj.bias"] = hf_state_dict["vivit.embeddings.patch_embeddings.projection.bias"]

  # Iteratively map weights for each layer.
  layer_indices = set()
  for k in hf_state_dict.keys():
    if k.startswith("vivit.layers."):
      # These keys have names of type "vivit.layers.{idx}.{component}.{weight/bias}".
      idx = int(k.split(".")[2])
      layer_indices.add(idx)

  for i in sorted(layer_indices):
    hf_prefix = f"vivit.layers.{i}." # Keys in the original model start with "vivit.layers.{i}".
    my_prefix = f"blocks.{i}." # Keys in the GaitViViT model start with "blocks.{i}".

    # Map weights for queries, keys and values.
    new_state_dict[f"{my_prefix}attn.q_proj.weight"] = hf_state_dict[f"{hf_prefix}attention.q_proj.weight"]
    new_state_dict[f"{my_prefix}attn.k_proj.weight"] = hf_state_dict[f"{hf_prefix}attention.k_proj.weight"]
    new_state_dict[f"{my_prefix}attn.v_proj.weight"] = hf_state_dict[f"{hf_prefix}attention.v_proj.weight"]

    if f"{hf_prefix}attention.q_proj.bias" in hf_state_dict:
      # Map biases if and only if they are defined for the GaitViViT model.
      new_state_dict[f"{my_prefix}attn.q_proj.bias"] = hf_state_dict[f"{hf_prefix}attention.q_proj.bias"]
      new_state_dict[f"{my_prefix}attn.k_proj.bias"] = hf_state_dict[f"{hf_prefix}attention.k_proj.bias"]
      new_state_dict[f"{my_prefix}attn.v_proj.bias"] = hf_state_dict[f"{hf_prefix}attention.v_proj.bias"]

    # Map weights for the attention output.
    new_state_dict[f"{my_prefix}attn.proj.weight"] = hf_state_dict[f"{hf_prefix}attention.o_proj.weight"]
    new_state_dict[f"{my_prefix}attn.proj.bias"] = hf_state_dict[f"{hf_prefix}attention.o_proj.bias"]

    # Map weights for layer normalization.
    new_state_dict[f"{my_prefix}norm1.weight"] = hf_state_dict[f"{hf_prefix}layernorm_before.weight"]
    new_state_dict[f"{my_prefix}norm1.bias"] = hf_state_dict[f"{hf_prefix}layernorm_before.bias"]
    new_state_dict[f"{my_prefix}norm2.weight"] = hf_state_dict[f"{hf_prefix}layernorm_after.weight"]
    new_state_dict[f"{my_prefix}norm2.bias"] = hf_state_dict[f"{hf_prefix}layernorm_after.bias"]

    # Map weights for the VivitMLP component.
    new_state_dict[f"{my_prefix}mlp.fc1.weight"] = hf_state_dict[f"{hf_prefix}mlp.fc1.weight"]
    new_state_dict[f"{my_prefix}mlp.fc1.bias"] = hf_state_dict[f"{hf_prefix}mlp.fc1.bias"]
    new_state_dict[f"{my_prefix}mlp.fc2.weight"] = hf_state_dict[f"{hf_prefix}mlp.fc2.weight"]
    new_state_dict[f"{my_prefix}mlp.fc2.bias"] = hf_state_dict[f"{hf_prefix}mlp.fc2.bias"]

  # Map weights for the final layer normalization before the classification head.
  new_state_dict["norm.weight"] = hf_state_dict["vivit.layernorm.weight"]
  new_state_dict["norm.bias"] = hf_state_dict["vivit.layernorm.bias"]

  # Note that head weights will not be mapped as the new classification head will be trained from scratch.

  return new_state_dict

**N.B.:** Further analysis revealed slight discrepancies of the order of $10^{-8}$ in the weight mapping procedure, although this is likely due to hardware limitations when working with floating-point arithmetic.